# 🏰 Deine Daten sind dein Burggraben

GPT-4o, Claude, Gemini — jeder kann diese Modelle nutzen. **Das Modell ist gemietet.** Was dich von der Konkurrenz unterscheidet, sind **deine Daten**.

In diesem Notebook siehst du den Unterschied:
1. Echte Ticket-Daten aus dem Projekt laden
2. Einen generischen Prompt darauf loslassen → mässiges Ergebnis
3. Mit deinen Daten tunen → deutlich besseres Ergebnis
4. Verstehen: **Domain-Wissen = Wettbewerbsvorteil**


In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, pandas as pd
from dspy_tasks.config import configure_dspy
from dspy_tasks.tasks import get_task
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *

MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)


## 1. Deine echten Daten

Wir laden echte Support-Tickets aus dem Projekt. Jedes Ticket hat:
- Eine **Zusammenfassung** (was ist das Problem?)
- Eine **Kategorie** (Network, Software, Hardware, ...)
- Eine **Priorität** (Critical, High, Medium, Low)
- Ein **zuständiges Team** (welches Team kümmert sich?)

Das Ziel: das Modell soll neue Tickets automatisch in die richtige Kategorie einsortieren, die Priorität setzen und das richtige Team zuweisen.


In [ ]:
task = get_task("ticket_routing")
examples = task.load_examples()

print(f"📊 {len(examples)} echte Tickets geladen\n")
for ex in examples[:5]:
    print(f"  📋 {str(ex.summary)[:80]}...")
    print(f"     → Kategorie: {ex.category} | Priorität: {ex.priority} | Team: {ex.assigned_group}")
    print()


## 2. Generischer Prompt → wie gut ist er?

Jetzt lassen wir das Modell diese Tickets klassifizieren — **ohne** Beispiele, **ohne** Domain-Wissen. Nur ein generischer Prompt: "Klassifiziere dieses Ticket."

Die Metrik bewertet drei Felder gleichzeitig:
- Priorität korrekt? → 40% Gewicht
- Kategorie korrekt? → 35% Gewicht  
- Richtiges Team? → 25% Gewicht


In [ ]:
print(f"⏳ Teste generischen Prompt auf {MODEL}...\n")

result = run_baseline("ticket_routing", max_eval=10)

display_score("Generischer Prompt (Baseline)", result.score)
display_results_table(result.individual_scores[:5])

print(f"\n👆 {result.score:.0%} — nicht schlecht, aber auch nicht gut genug für Produktion.")
print(f"   Schau dir die Fehler an: das Modell kennt eure Teams und Kategorien nicht!")


## 3. Domain-Tuning → der Unterschied

Jetzt lassen wir den Optimizer mit **deinen echten Ticket-Beispielen** trainieren. Er sucht automatisch die besten Few-Shot-Beispiele aus deinen Daten und baut einen besseren Prompt.

Gleiches Modell, gleiche Aufgabe — aber mit Domain-Wissen.


In [ ]:
print(f"⏳ Optimiere mit echten Ticket-Daten... (10-30 Sekunden)\n")

result = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=10)

display_improvement(result.baseline_score, result.optimized_score)
display_prompt_diff(result.prompt_before, result.prompt_after)


## 4. Was du gerade gesehen hast

| | Generisch | Domain-getuned |
|---|---|---|
| Prompt | "Klassifiziere dieses Ticket" | Optimierter Prompt + echte Beispiele |
| Wissen | Keines über eure Teams/Kategorien | Lernt aus euren echten Tickets |
| Aufwand | 0 Sekunden | 10-30 Sekunden (einmalig) |

### 💡 Die Lektion

- **Das Modell ist gemietet** — GPT-4o kann jeder nutzen
- **Deine Daten gehören dir** — eure Tickets, eure Kategorien, eure Teams
- **Tuning macht den Unterschied** — messbar, reproduzierbar, einmalige Kosten
- **Das ist dein Burggraben** — kein Konkurrent kann eure Trainingsdaten kopieren


In [ ]:
display_insight("Dein Burggraben",
    f"Generischer Prompt: {result.baseline_score:.0%}. "
    f"Getuned mit DEINEN Daten: {result.optimized_score:.0%}. "
    "Das Modell ist gemietet — deine Daten sind es nicht.")


## ⏭️ Weiter geht's!

Deine Daten + Tuning = Burggraben. Aber können auch **Agenten** optimiert werden? Agenten die selbst entscheiden, welche Tools sie nutzen?

👉 **[Weiter zu Notebook: Agenten →](04_agents.ipynb)**
